## Overall plan

- what we currently have Bellhop : train XGBoost
- now, we want to see how adding high dimensional bathymetry data and SSP data (from GEBCO and WOA data) improves overall modelling of the residual
- in order to learn a dense/low-dimensional representation of bathymetry data + SSP data, we will use an encoder
- then we will predict residual using MLP

(Optional) 0. Train autoencoder (for practice more than anything)

Putting a minimum viable implementation for MNIST below. You'll need to adapt to your own dataset.

In [ ]:
# Train autoencoder (bathymetry data and SSP separately!)

# data input = bathymetry data
# data output = reconstructed(bathymetry data)

import torch, torch.nn as nn
from torchvision import datasets, transforms

# images! your data will be bathymetry data + SSP data
# TODO Determine what your data loader for bathymetry data and SSP data should look like
data = datasets.MNIST(".", train=True, download=True, transform=transforms.ToTensor())
loader = torch.utils.data.DataLoader(data, batch_size=256,shuffle=True)

# class inheritance
class AutoEncoder(nn.Module):
    def __init__(self, latent=32):
        super().__init__()
        # TODO Understand what the enc and dec looks like, what RELU and Sigmoid functions does etc 
        # and determine what your encoder and decoder should look like 
        self.enc = nn.Sequential(
            nn.Linear(784,128), nn.ReLU(),
            nn.Linear(128, latent)
        )
        self.dec = nn.Sequential(
            nn.Linear(latent, 128), nn.ReLU(), # look up RELU and Sigmoid function is doing 
            nn.Linear(128,784), nn.Sigmoid()
        )

    def forward(self, x):
        return self.dec(self.enc(x))

model = AutoEncoder()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss() # takes in argument of model output, target

for epoch in range(10):
    for x, _ in loader:
        x = x.view(x.size(0), -1) # for images, you'll need to flatten to 784 (2D -> 1D) but your data may already be flattened
        loss = criterion(model(x), x) # (model output, target)
        # using adam optimizer
        # TODO understand the steps below!
        opt.zero_grad(); loss.backward(); opt.step() # watch 3B1B video on backpropagation!

    print(f"epoch {epoch} loss {loss.item():.4f}")



1.  Train encoder + MLP together 

In [ ]:
from __future__ import annotations

import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch  # uv pip install torch
import torch.nn as nn
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import ocean
import propagation as prop

# ── Paths ─────────────────────────────────────────────────────────────────────
try:
    HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    HERE = os.getcwd()
DATA_ROOT = os.path.join(HERE, "Data")
DATA_PATH = os.path.join(DATA_ROOT, "BellhopData", "bellhop_monthly_original.csv")
CACHE_PATH = os.path.join(DATA_ROOT, "BellhopData", "profiles_cache.npz")
MODEL_PATH = os.path.join(DATA_ROOT, "BellhopData", "tl_residual_mlp.pt")
GEBCO_PATH = os.path.join(DATA_ROOT, "GEBCO_01_Jun_2026_cd11525db157",
                          "gebco_2026_n72.0_s62.0_w-45.0_e-10.0.nc")
TEMP_DIR = os.path.join(DATA_ROOT, "woa23_t_B5C2_1.00_csv")
SAL_DIR = os.path.join(DATA_ROOT, "woa23_s_B5C2_1.00_csv")

RESULTS_DIR = os.path.join(HERE, "Results")
os.makedirs(RESULTS_DIR, exist_ok=True)

LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = 62.0, 70.0, -44.0, -13.0  # Denmark Strait

# ── Low-dimensional (tabular) features — same set fit_xgb.ipynb uses ───────────
NUMERIC = [
    "range_km", "log10_freq_hz",
    "src_seabed_depth_m", "rcv_seabed_depth_m",
    "path_min_depth_m", "path_mean_depth_m",
    "src_depth_m", "rcv_depth_m", "layer_mean_speed_ms",
]
BINARY = ["is_shadow", "month_sin", "month_cos"]
LAYERS = ["surface", "mid", "deep"]          # one-hot encoded for the MLP

# ── High-dimensional profile features ─────────────────────────────────────────
N_BATHY = 64                    # seabed samples along the source→receiver path
SSP_DEPTHS = prop.WOA_DEPTHS_M  # 57 standard WOA levels — already a fixed grid
N_SSP = len(SSP_DEPTHS)

# ── Model / training hyper-parameters ─────────────────────────────────────────
LATENT_DIM = 8      # size of each encoder's bottleneck (bathymetry, SSP)
HIDDEN = 128
EPOCHS = 300
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-4
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = "cpu"

In [ ]:
# Save results in a unique directory (same convention as fit_xgb.ipynb)
from coolname import generate_slug  # uv pip install coolname


def make_unique_dir():
    unique_dir = os.path.join(RESULTS_DIR, generate_slug(2))
    os.makedirs(unique_dir, exist_ok=True)
    return unique_dir


def load_and_prepare() -> pd.DataFrame:
    """Load the Bellhop dataset, drop artifacts, add target + engineered features."""
    df = pd.read_csv(DATA_PATH)
    valid = (df["tl_bellhop_db"].notna() & np.isfinite(df["tl_bellhop_db"])
             & (df["tl_bellhop_db"] <= 160.0))
    print(f"Loaded {len(df)} rows; dropped {(~valid).sum()} invalid Bellhop values "
          f"-> {valid.sum()} remain")
    df = df[valid].copy()

    df["residual_db"] = df["tl_bellhop_db"] - df["tl_analytic_db"]
    df["log10_freq_hz"] = np.log10(df["freq_hz"])
    df["is_shadow"] = (df["shadow_penalty_db"] > 0).astype(float)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

    if "group_id" not in df.columns:
        df["group_id"] = df["source_file"].astype(str) + "_" + df["pair_i"].astype(str)
    return df

In [ ]:
# ── High-dimensional inputs: one bathymetry + one SSP profile per source-receiver
# pair. Geometry (and therefore both profiles) is constant within a `group_id`, so
# they are built once per group and cached to disk — the WOA load is the slow part.

def _ffill(a: np.ndarray) -> np.ndarray:
    """Forward-fill NaNs (land / below the deepest valid level); back-fill the head."""
    a = np.asarray(a, dtype=np.float64).copy()
    ok = np.where(np.isfinite(a))[0]
    if ok.size == 0:
        return np.zeros_like(a)
    a[:ok[0]] = a[ok[0]]
    for k in range(1, len(a)):
        if not np.isfinite(a[k]):
            a[k] = a[k - 1]
    return a


def path_bathy_vector(lat1, lon1, lat2, lon2, rgi, n=N_BATHY) -> np.ndarray:
    """Seabed depth (m) at `n` equally spaced points along the great-circle path."""
    range_km = ocean.great_circle_km(lat1, lon1, lat2, lon2)
    brg = ocean.bearing_between(lat1, lon1, lat2, lon2)
    pts = np.array([ocean.point_at_bearing(lat1, lon1, brg, float(r))
                    for r in np.linspace(0.0, range_km, n)])
    return _ffill(rgi(pts))


def build_profiles(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, dict[str, int]]:
    """
    (bathy, ssp, group_id -> row index) for every unique pair in `df`.

    bathy : (n_groups, N_BATHY) seabed depth along the path, m
    ssp   : (n_groups, N_SSP)   sound speed at the source on the WOA levels, m/s
    """
    gids = sorted(df["group_id"].unique())
    index = {g: i for i, g in enumerate(gids)}

    if os.path.exists(CACHE_PATH):
        z = np.load(CACHE_PATH, allow_pickle=False)
        if list(z["gids"]) == gids:
            print(f"Loaded cached profiles for {len(gids)} pairs <- {CACHE_PATH}")
            return z["bathy"], z["ssp"], index

    print("Loading GEBCO bathymetry...")
    depth_rgi = ocean.gebco_rgi(GEBCO_PATH)
    print("Loading WOA23 climatology (slow, cached afterwards)...")
    woa = prop.load_woa(TEMP_DIR, SAL_DIR, LAT_MIN, LAT_MAX, LON_MIN, LON_MAX)

    first = df.drop_duplicates("group_id").set_index("group_id")
    bathy = np.zeros((len(gids), N_BATHY))
    ssp = np.zeros((len(gids), N_SSP))
    for g, i in index.items():
        r = first.loc[g]
        bathy[i] = path_bathy_vector(r["src_lat"], r["src_lon"],
                                     r["rcv_lat"], r["rcv_lon"], depth_rgi)
        c, _ = prop.sound_speed_profile(woa, r["src_lat"], r["src_lon"], int(r["month"]))
        ssp[i] = _ffill(c)
        if (i + 1) % 500 == 0:
            print(f"  {i + 1}/{len(gids)} pairs")

    np.savez_compressed(CACHE_PATH, gids=np.array(gids), bathy=bathy, ssp=ssp)
    print(f"Saved profiles -> {CACHE_PATH}")
    return bathy, ssp, index

In [ ]:
# ── Model: two profile encoders + a shared MLP head, trained jointly ──────────
# The encoders are *not* pretrained — their bottlenecks are learned end-to-end from
# the residual loss, so they compress bathymetry/SSP into whatever the head needs.

class ProfileEncoder(nn.Module):
    """Compress a high-dimensional profile down to `n_latent` numbers."""

    def __init__(self, n_in: int, n_latent: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, n_latent),
        )

    def forward(self, x):
        return self.net(x)


class ResidualNet(nn.Module):
    """[tabular | encode(bathy) | encode(ssp)] -> MLP -> residual (standardised)."""

    def __init__(self, n_tab: int, n_bathy: int, n_ssp: int,
                 n_latent: int = LATENT_DIM, hidden: int = HIDDEN):
        super().__init__()
        self.enc_bathy = ProfileEncoder(n_bathy, n_latent)
        self.enc_ssp = ProfileEncoder(n_ssp, n_latent)
        self.head = nn.Sequential(
            nn.Linear(n_tab + 2 * n_latent, hidden), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, 1),
        )

    def forward(self, tab, bathy, ssp):
        z = torch.cat([tab, self.enc_bathy(bathy), self.enc_ssp(ssp)], dim=1)
        return self.head(z).squeeze(1)


def _standardise(train: np.ndarray, *others: np.ndarray):
    """Z-score using train statistics only; returns (scaled arrays..., mean, std)."""
    mu, sd = train.mean(axis=0), train.std(axis=0)
    sd = np.where(sd < 1e-8, 1.0, sd)
    return [(a - mu) / sd for a in (train, *others)], mu, sd


def _metrics(y_true: np.ndarray, y_pred: np.ndarray, label: str) -> None:
    print(f"  {label:6s} R2={r2_score(y_true, y_pred):.4f}  "
          f"RMSE={mean_squared_error(y_true, y_pred) ** 0.5:.3f} dB  "
          f"MAE={mean_absolute_error(y_true, y_pred):.3f} dB")

In [ ]:
def run() -> None:
    df = load_and_prepare()
    bathy_g, ssp_g, gindex = build_profiles(df)

    # ── Assemble the three input blocks ───────────────────────────────────────
    tab = df[NUMERIC + BINARY].to_numpy(dtype=np.float64)
    onehot = np.stack([(df["layer"] == L).to_numpy(dtype=np.float64) for L in LAYERS], 1)
    tab = np.hstack([tab, onehot])
    rows = df["group_id"].map(gindex).to_numpy()      # row -> profile index
    bathy = bathy_g[rows]
    ssp = ssp_g[rows]
    y = df["residual_db"].to_numpy(dtype=np.float64)
    groups = df["group_id"]

    # ── Group-wise train / val / test split (pairs never straddle a split) ────
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    fit_idx, test_idx = next(gss.split(tab, y, groups=groups))
    gss_val = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    sub_train, sub_val = next(gss_val.split(tab[fit_idx], y[fit_idx],
                                            groups=groups.iloc[fit_idx]))
    train_idx, val_idx = fit_idx[sub_train], fit_idx[sub_val]
    for name, idx in [("Train", train_idx), ("Val", val_idx), ("Test", test_idx)]:
        print(f"{name:5s}: {len(idx)} rows, {groups.iloc[idx].nunique()} pairs")

    # ── Standardise inputs and target on train statistics only ────────────────
    med = np.nanmedian(tab[train_idx], axis=0)
    tab = np.where(np.isfinite(tab), tab, med)        # layer_mean_speed_ms has NaNs
    (tab_tr, tab_va, tab_te), *_ = _standardise(tab[train_idx], tab[val_idx], tab[test_idx])
    (bat_tr, bat_va, bat_te), *_ = _standardise(bathy[train_idx], bathy[val_idx], bathy[test_idx])
    (ssp_tr, ssp_va, ssp_te), *_ = _standardise(ssp[train_idx], ssp[val_idx], ssp[test_idx])
    y_mu, y_sd = y[train_idx].mean(), y[train_idx].std()

    def tensors(t, b, s, idx):
        return (torch.tensor(t, dtype=torch.float32, device=DEVICE),
                torch.tensor(b, dtype=torch.float32, device=DEVICE),
                torch.tensor(s, dtype=torch.float32, device=DEVICE),
                torch.tensor((y[idx] - y_mu) / y_sd, dtype=torch.float32, device=DEVICE))

    Xtr = tensors(tab_tr, bat_tr, ssp_tr, train_idx)
    Xva = tensors(tab_va, bat_va, ssp_va, val_idx)
    Xte = tensors(tab_te, bat_te, ssp_te, test_idx)

    # ── Train encoders + head jointly ─────────────────────────────────────────
    model = ResidualNet(tab.shape[1], N_BATHY, N_SSP).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.MSELoss()
    n = len(train_idx)

    best_val, best_state, history = np.inf, None, []
    print("\nTraining encoder + MLP...")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = torch.randperm(n, device=DEVICE)
        for k in range(0, n, BATCH_SIZE):
            b = perm[k:k + BATCH_SIZE]
            opt.zero_grad()
            loss = loss_fn(model(Xtr[0][b], Xtr[1][b], Xtr[2][b]), Xtr[3][b])
            loss.backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_rmse = float(loss_fn(model(*Xva[:3]), Xva[3]).sqrt()) * y_sd
            train_rmse = float(loss_fn(model(*Xtr[:3]), Xtr[3]).sqrt()) * y_sd
        history.append((epoch, train_rmse, val_rmse))
        if val_rmse < best_val:
            best_val = val_rmse
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        if epoch % 25 == 0 or epoch == 1:
            print(f"  epoch {epoch:3d}  train RMSE {train_rmse:6.3f} dB  "
                  f"val RMSE {val_rmse:6.3f} dB")

    model.load_state_dict(best_state)   # early stopping on the validation split
    print(f"Best val RMSE {best_val:.3f} dB")

    # ── Evaluate ──────────────────────────────────────────────────────────────
    model.eval()
    with torch.no_grad():
        preds = {name: (model(*X[:3]).cpu().numpy() * y_sd + y_mu)
                 for name, X in [("train", Xtr), ("val", Xva), ("test", Xte)]}
    print("Performance:")
    for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
        _metrics(y[idx], preds[name], name)

    torch.save({"state_dict": model.state_dict(),
                "y_mu": y_mu, "y_sd": y_sd,
                "n_tab": tab.shape[1], "latent_dim": LATENT_DIM}, MODEL_PATH)
    print(f"Saved model -> {MODEL_PATH}")

    _plot(df.iloc[test_idx].copy(), y[test_idx], preds["test"], history)


def _plot(df_test, y_test, y_pred, history) -> None:
    out_dir = make_unique_dir()
    colors = {"surface": "#2166ac", "mid": "#4dac26", "deep": "#d6604d"}
    df_test["_pred"] = y_pred

    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    for layer, grp in df_test.groupby("layer"):
        ax.scatter(grp["residual_db"], grp["_pred"], s=10, alpha=0.5,
                   color=colors.get(layer, "gray"), label=layer)
    lims = [min(y_test.min(), y_pred.min()) - 2, max(y_test.max(), y_pred.max()) + 2]
    ax.plot(lims, lims, "k--", lw=0.8, label="1:1")
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("Actual residual (dB)"); ax.set_ylabel("Predicted residual (dB)")
    ax.set_title("Encoder + MLP residual — test set"); ax.legend(markerscale=2)
    ax.grid(True, lw=0.3, alpha=0.4)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "mlp_actual_vs_predicted.png"), dpi=150)
    print(f"Saved {os.path.join(out_dir, 'mlp_actual_vs_predicted.png')}")

    ep, tr, va = zip(*history)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(ep, tr, label="train", color="#377eb8")
    ax.plot(ep, va, label="val", color="#d6604d")
    ax.set_xlabel("Epoch"); ax.set_ylabel("RMSE (dB)")
    ax.set_title("Training curve"); ax.legend()
    ax.grid(True, lw=0.3, alpha=0.4)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "mlp_training_curve.png"), dpi=150)
    print(f"Saved {os.path.join(out_dir, 'mlp_training_curve.png')}")
    plt.close("all")


run()